## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import FuncFormatter

# Plot styling
sns.set(style="whitegrid", font_scale=1.2)
sns.set_context("talk")

## Load Matches & Teams

In [2]:
matches = pd.read_parquet("Datasets/SkillCorner Premier League 24-25 data/matches_clean.parquet")

team_lookup = pd.concat([
    matches[["home_team_id","home_team_name"]].rename(columns={"home_team_id":"team_id","home_team_name":"team_name"}),
    matches[["away_team_id","away_team_name"]].rename(columns={"away_team_id":"team_id","away_team_name":"team_name"})
]).drop_duplicates()

## Load Events Data

In [3]:
folder = Path("Datasets/SkillCorner Premier League 24-25 data/dynamic_events_pl_24/dynamic")
dfs = [pd.read_parquet(file) for file in folder.glob("*.parquet")]
events = pd.concat(dfs, ignore_index=True)

print(f"Total events: {len(events)}, Unique matches: {events['match_id'].nunique()}")
print(events["event_type"].value_counts().head(10))

Total events: 1811078, Unique matches: 378
event_type
passing_option        939059
player_possession     362853
on_ball_engagement    326100
off_ball_run          183066
Name: count, dtype: int64


## Load Player Match Data

In [4]:
players_match = pd.read_parquet("Datasets/SkillCorner Premier League 24-25 data/players_match.parquet")

players_match["player_id"] = players_match["id"]
players_match["player_name"] = players_match["short_name"]
players_match["position"] = players_match["player_role_acronym"]
players_match["position_group"] = players_match["player_role_position_group"]

# Minutes summary
minutes_summary = (
    players_match.groupby(["player_id","player_name"])
    .agg(
        minutes_tip=("playing_time_total_minutes_tip","sum"),
        minutes_otip=("playing_time_total_minutes_otip","sum"),
        minutes_played=("playing_time_total_minutes_played","sum")
    )
    .reset_index()
)

# Main position
position_minutes = (
    players_match.groupby(["player_id","player_name","position","position_group"], as_index=False)
    .agg(minutes_played=("playing_time_total_minutes_played","sum"))
)

position_minutes_sorted = position_minutes.sort_values(["player_id","minutes_played"], ascending=[True,False])

def get_main_position(df):
    main_pos = df.iloc[0]["position"]
    pos_group = df.iloc[0]["position_group"]

    if main_pos == "SUB" and len(df) > 1:
        main_pos = df.iloc[1]["position"]
        pos_group = df.iloc[1]["position_group"]

    if main_pos == "GK":
        pos_group = "Goalkeeper"

    return pd.Series({"main_position": main_pos, "position_group": pos_group})

main_positions = position_minutes_sorted.groupby("player_id").apply(get_main_position).reset_index()

# Merge final player dataset
players = minutes_summary.merge(main_positions, on="player_id", how="left")
players["minutes_played"] = players["minutes_played"].replace(0, np.nan)

C:\Users\vicky\AppData\Local\Temp\ipykernel_28412\2970160260.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  main_positions = position_minutes_sorted.groupby("player_id").apply(get_main_position).reset_index()


## Passing Options & Values (xPass × xThreat)

In [5]:
passing_options = events[events["event_type"] == "passing_option"].copy()

passing_options["passer_id"] = passing_options["player_in_possession_id"]
passing_options["passer_name"] = passing_options["player_in_possession_name"]
passing_options["receiver_id"] = passing_options["player_id"]
passing_options["receiver_name"] = passing_options["player_name"]

passing_options["pass_value"] = passing_options["xthreat"] * passing_options["xpass_completion"]

best_option = (
    passing_options.groupby(["match_id","associated_player_possession_event_id"], as_index=False)
    .agg(best_pass_value=("pass_value","max"))
)
passing_options = passing_options.merge(best_option, on=["match_id","associated_player_possession_event_id"], how="left")

# Keep only possessions with >1 passing option
option_counts = passing_options.groupby(["match_id","associated_player_possession_event_id"]).size().reset_index(name="n_options")
passing_options = passing_options.merge(option_counts, on=["match_id","associated_player_possession_event_id"], how="left")
passing_options = passing_options[passing_options["n_options"] > 1]

# Extract actual passes
chosen_passes = passing_options[passing_options["targeted"] == True].copy()

## Decision & Execution Metrics

In [6]:
# Decision quality
chosen_passes["decision_quality"] = chosen_passes["pass_value"] / chosen_passes["best_pass_value"].replace(0,np.nan)
chosen_passes["chose_best"] = chosen_passes["pass_value"] >= chosen_passes["best_pass_value"] * 0.95

# Execution
chosen_passes["actual_completion"] = (chosen_passes["received"] == True).astype(int)
chosen_passes["completion_minus_xpass"] = chosen_passes["actual_completion"] - chosen_passes["xpass_completion"]

## Passer Metrics (Per Player)

In [7]:
passer_metrics = (
    chosen_passes.groupby(["passer_id","passer_name","team_shortname"])
    .agg(
        passes_attempted=("pass_value","count"),
        avg_decision_quality=("decision_quality","mean"),
        chose_best_rate=("chose_best","mean"),
        chose_best_count=("chose_best","sum"),
        chose_not_best_count=("chose_best", lambda x: (~x).sum()),
        completion_minus_xpass_per_pass=("completion_minus_xpass","mean"),
        completion_minus_xpass_total=("completion_minus_xpass","sum"),
        total_pass_value=("pass_value","sum"),
        avg_pass_value=("pass_value","mean"),
        avg_xthreat=("xthreat","mean"),
        avg_xpass_completion=("xpass_completion","mean")
    )
    .reset_index()
).merge(players, left_on="passer_id", right_on="player_id", how="left")

# Per 90 normalisation
passer_metrics["passes_per90_tip"] = passer_metrics["passes_attempted"] / passer_metrics["minutes_tip"] * 90
passer_metrics["pass_value_per90_tip"] = passer_metrics["total_pass_value"] / passer_metrics["minutes_tip"] * 90

# Filter for players with enough minutes
full_games = 5
minutes_threshold = full_games * 90
passers_filtered = passer_metrics[(passer_metrics["passes_attempted"] >= 200) & (passer_metrics["minutes_played"] >= minutes_threshold)]

## Receiver Metrics

In [8]:
receiver_metrics = (
    passing_options.groupby(["receiver_id","receiver_name","team_shortname"])
    .agg(
        passing_options_available=("receiver_id","count"),
        times_targeted=("targeted","sum"),
        option_selection_rate=("targeted","mean"),
        total_option_value=("pass_value","sum"),
        avg_option_value=("pass_value","mean"),
        total_xthreat_available=("xthreat","sum"),
        avg_xthreat_available=("xthreat","mean"),
        avg_xpass_passing_option_created=("xpass_completion","mean")
    )
    .reset_index()
).merge(players, left_on="receiver_id", right_on="player_id", how="left")

receiver_metrics["options_available_per90_tip"] = receiver_metrics["passing_options_available"] / receiver_metrics["minutes_tip"] * 90
receiver_metrics["targets_received_per90_tip"] = receiver_metrics["times_targeted"] / receiver_metrics["minutes_tip"] * 90
receiver_metrics["option_value_per90_tip"] = receiver_metrics["total_option_value"] / receiver_metrics["minutes_tip"] * 90
receiver_metrics["xthreat_available_per90_tip"] = receiver_metrics["total_xthreat_available"] / receiver_metrics["minutes_tip"] * 90

receivers_filtered = receiver_metrics[(receiver_metrics["passing_options_available"] >= 200) & (receiver_metrics["minutes_played"] >= minutes_threshold)]

## Off-Ball Run Metrics (New)

In [12]:
off_ball_runs = events[events["event_type"] == "off_ball_run"].copy()

# Drop columns that are all NaN
off_ball_runs = off_ball_runs.dropna(axis=1, how='all')

pd.set_option('display.max_columns', None)  # ensure all columns are visible
print("Columns remaining:", len(off_ball_runs.columns))
print(off_ball_runs.columns)

Columns remaining: 118
Index(['event_id', 'index', 'match_id', 'frame_start', 'frame_end',
       'time_start', 'time_end', 'minute_start', 'second_start', 'duration',
       ...
       'intended_run_behind', 'push_defensive_line', 'break_defensive_line',
       'passing_option_at_start', 'n_opponents_ahead_end',
       'n_opponents_ahead_start', 'n_opponents_overtaken',
       'is_player_possession_start_matched',
       'is_player_possession_end_matched', 'is_pass_reception_matched'],
      dtype='object', length=118)


In [17]:
off_ball_runs.head(5)

,event_id,index,match_id,frame_start,frame_end,time_start,time_end,minute_start,second_start,duration,period,attacking_side_id,attacking_side,event_type_id,event_type,event_subtype_id,event_subtype,player_id,player_name,player_position_id,player_position,player_in_possession_id,player_in_possession_name,player_in_possession_position_id,player_in_possession_position,team_id,team_shortname,x_start,y_start,channel_id_start,channel_start,third_id_start,third_start,penalty_area_start,x_end,y_end,channel_id_end,channel_end,third_id_end,third_end,penalty_area_end,associated_player_possession_event_id,associated_player_possession_frame_start,game_state_id,game_state,team_score,opponent_team_score,phase_index,team_in_possession_phase_type_id,team_in_possession_phase_type,team_out_of_possession_phase_type_id,team_out_of_possession_phase_type,lead_to_shot,lead_to_goal,distance_covered,trajectory_angle,trajectory_direction_id,trajectory_direction,in_to_out,out_to_in,speed_avg,speed_avg_band_id,speed_avg_band,separation_start,separation_end,separation_gain,last_defensive_line_x_start,last_defensive_line_x_end,delta_to_last_defensive_line_start,delta_to_last_defensive_line_end,delta_to_last_defensive_line_gain,last_defensive_line_height_start,last_defensive_line_height_end,last_defensive_line_height_gain,inside_defensive_shape_start,inside_defensive_shape_end,location_to_player_in_possession_id_start,location_to_player_in_possession_start,location_to_player_in_possession_id_end,location_to_player_in_possession_end,distance_to_player_in_possession_start,distance_to_player_in_possession_end,player_in_possession_x_start,player_in_possession_y_start,player_in_possession_channel_id_start,player_in_possession_channel_start,player_in_possession_third_id_start,player_in_possession_third_start,player_in_possession_penalty_area_start,player_in_possession_x_end,player_in_possession_y_end,player_in_possession_channel_id_end,player_in_possession_channel_end,player_in_possession_third_id_end,player_in_possession_third_end,player_in_possession_penalty_area_end,targeted,received,received_in_space,dangerous,difficult_pass_target,xthreat,xpass_completion,passing_option_score,predicted_passing_option,passing_option_at_player_possession_start,n_simultaneous_runs,give_and_go,intended_run_behind,push_defensive_line,break_defensive_line,passing_option_at_start,n_opponents_ahead_end,n_opponents_ahead_start,n_opponents_overtaken,is_player_possession_start_matched,is_player_possession_end_matched,is_pass_reception_matched
1,1_0,1,1650385,59,72,00:04.9,00:06.2,0,4,1.3,1,2,right_to_left,1,off_ball_run,2.0,coming_short,12941,E. Smith Rowe,19,CF,13068.0,S. Lukić,12.0,LM,48,Fulham,-2.99,8.19,3,center,2,middle_third,False,-6.23,2.77,3,center,2,middle_third,False,8_1,62.0,3,drawing,0,0,0,1,create,9,medium_block,False,False,5.91,-120.87,4.0,sideway_right,False,False,17.30,2.0,running,1.58,1.95,0.37,19.60,19.32,22.59,25.55,2.96,32.90,33.18,0.28,False,True,3.0,ahead,3.0,ahead,10.59,5.41,-9.58,-0.10,3.0,center,2.0,middle_third,False,-9.72,-1.36,3.0,center,2.0,middle_third,False,False,False,None,False,False,0.0013,0.9808,0.7928,True,True,0.0,False,None,None,None,True,9.0,10.0,1.0,True,True,None
15,1_1,15,1650385,125,156,00:11.5,00:14.6,0,11,3.1,1,2,right_to_left,1,off_ball_run,9.0,support,5794,K. Tete,8,RB,12168.0,Adama Traoré,17.0,RW,48,Fulham,-12.74,-29.71,5,wide_right,2,middle_third,False,3.55,-28.08,5,wide_right,2,middle_third,False,8_3,143.0,3,drawing,0,0,0,1,create,9,medium_block,False,False,16.03,5.71,1.0,forward,False,False,19.00,2.0,running,4.65,4.90,0.25,13.68,18.60,26.42,15.05,-11.37,38.82,33.90,-4.92,False,False,1.0,behind,1.0,behind,22.27,7.65,9.35,-32.52,5.0,wide_right,2.0,middle_third,False,10.91,-26.00,5.0,wide_right,2.0,middle_third,False,False,False,None,False,False,0.0020,0.9211,0.9390,True,True,1.0,False,None,None,None,True,6.0,8.0,2.0,True,True,None
20,1_2,20,1650385,145,159,00:13.5,00:14.9,0,13,1.4,1,2,right_to_left,1,off_ball_run,9.0,support,11438,Andreas

In [ ]:
# Aggregate off-ball metrics per player
player_offball_metrics = (
    off_ball_runs.groupby(["player_id", "player_name"])
    .agg(
        offball_runs_total=("event_id", "count"),
        offball_distance_total=("distance_covered", "sum"),
        offball_distance_avg=("distance_covered", "mean"),
        offball_speed_avg=("speed_avg", "mean")
    )
    .reset_index()
)

# Merge minutes played from players DataFrame
player_offball_metrics = player_offball_metrics.merge(
    players[['player_id', 'minutes_played']], on='player_id', how='left'
)

## Merge Passing + Off-Ball Metrics

In [ ]:
full_player_metrics = passers_filtered.merge(
    player_offball_metrics,
    left_on="passer_id",
    right_on="player_id",
    how="left"
)

In [ ]:
full_player_metrics.head()

## On-ball engagement

In [ ]:
on_ball_engagement = events[events["event_type"] == "on_ball_engagement"].copy()

# Drop columns that are all NaN
on_ball_engagement = on_ball_engagement.dropna(axis=1, how='all')

pd.set_option('display.max_columns', None)  # ensure all columns are visible
print("Columns remaining:", len(on_ball_engagement.columns))

In [ ]:
on_ball_engagement.head(10)